In [1]:
import pandas as pd
import numpy as np
import os

RAW_PATH = r"C:\Users\priya\OneDrive\Desktop\Documents\Downloads\Olist_Data_Warehouse\Dataset\Raw"
PROCESSED_PATH = r"C:\Users\priya\OneDrive\Desktop\Documents\Downloads\Olist_Data_Warehouse\Dataset\Processed"

In [3]:
# Load original datasets
orders = pd.read_csv(os.path.join(RAW_PATH, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(RAW_PATH, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(RAW_PATH, "olist_customers_dataset.csv"))
products = pd.read_csv(os.path.join(RAW_PATH, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(RAW_PATH, "olist_sellers_dataset.csv"))
payments = pd.read_csv(os.path.join(RAW_PATH, "olist_order_payments_dataset.csv"))
reviews = pd.read_csv(os.path.join(RAW_PATH, "olist_order_reviews_dataset.csv"))
geolocation = pd.read_csv(os.path.join(RAW_PATH, "olist_geolocation_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_PATH, "product_category_name_translation.csv"))

# Load cleaned datasets
orders_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "orders_clean.csv"))
order_items_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "order_items_clean.csv"))
customers_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "customers_clean.csv"))
products_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "products_clean.csv"))
sellers_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "sellers_clean.csv"))
payments_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "payments_clean.csv"))
reviews_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "reviews_clean.csv"))
geolocation_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "geolocation_clean.csv"))
category_translation_clean = pd.read_csv(os.path.join(PROCESSED_PATH, "category_translation_clean.csv"))

print("All 18 datasets loaded successfully!")

All 18 datasets loaded successfully!


In [4]:
# Create a dictionary containing original and cleaned datasets
datasets = {
    "Orders": (orders, orders_clean),
    "Order Items": (order_items, order_items_clean),
    "Customers": (customers, customers_clean),
    "Products": (products, products_clean),
    "Sellers": (sellers, sellers_clean),
    "Payments": (payments, payments_clean),
    "Reviews": (reviews, reviews_clean),
    "Geolocation": (geolocation, geolocation_clean),
    "Category Translation": (category_translation, category_translation_clean)
}

# Create summary
summary = []

for name, (original, cleaned) in datasets.items():
    summary.append({
        "Dataset": name,
        "Original Rows": len(original),
        "Cleaned Rows": len(cleaned),
        "Rows Removed": len(original) - len(cleaned),
        "Original Columns": len(original.columns),
        "Cleaned Columns": len(cleaned.columns)
    })

# Convert summary into a DataFrame
row_summary = pd.DataFrame(summary)

row_summary

,Dataset,Original Rows,Cleaned Rows,Rows Removed,Original Columns,Cleaned Columns
0,Orders,99441,99441,0,8,8
1,Order Items,112650,112650,0,7,7
2,Customers,99441,99441,0,5,5
3,Products,32951,32951,0,9,9
4,Sellers,3095,3095,0,4,4
5,Payments,103886,103886,0,5,5
6,Reviews,99224,99224,0,7,7
7,Geolocation,1000163,738332,261831,5,5
8,Category Translation,71,71,0,2,2


In [5]:
missing_summary = []

for name, (original, cleaned) in datasets.items():
    missing_summary.append({
        "Dataset": name,
        "Original Missing Values": original.isnull().sum().sum(),
        "Cleaned Missing Values": cleaned.isnull().sum().sum()
    })

missing_summary_df = pd.DataFrame(missing_summary)

missing_summary_df

,Dataset,Original Missing Values,Cleaned Missing Values
0,Orders,4908,4908
1,Order Items,0,0
2,Customers,0,0
3,Products,2448,2448
4,Sellers,0,0
5,Payments,0,0
6,Reviews,145903,145903
7,Geolocation,0,0
8,Category Translation,0,0


In [6]:
primary_key_checks = {
    "Orders": ("order_id", orders_clean),
    "Order Items": ("order_id + order_item_id", order_items_clean),
    "Customers": ("customer_id", customers_clean),
    "Products": ("product_id", products_clean),
    "Sellers": ("seller_id", sellers_clean),
    "Payments": ("order_id + payment_sequential", payments_clean),
    "Reviews": ("review_id", reviews_clean),
    "Geolocation": ("No single primary key", geolocation_clean),
    "Category Translation": ("product_category_name", category_translation_clean)
}

for name, (key, df) in primary_key_checks.items():
    if name == "Geolocation":
        print(name, "-", key)
    elif name == "Order Items":
        print(name, "- Duplicate keys:",
              df.duplicated(subset=["order_id", "order_item_id"]).sum())
    elif name == "Payments":
        print(name, "- Duplicate keys:",
              df.duplicated(subset=["order_id", "payment_sequential"]).sum())
    else:
        print(name, "- Duplicate keys:",
              df.duplicated(subset=[key]).sum())

Orders - Duplicate keys: 0
Order Items - Duplicate keys: 0
Customers - Duplicate keys: 0
Products - Duplicate keys: 0
Sellers - Duplicate keys: 0
Payments - Duplicate keys: 0
Reviews - Duplicate keys: 814
Geolocation - No single primary key
Category Translation - Duplicate keys: 0


In [7]:
# Find all records with repeated review IDs
duplicate_reviews = reviews_clean[
    reviews_clean.duplicated(subset=["review_id"], keep=False)
].sort_values("review_id")

print("Records with repeated review IDs:", len(duplicate_reviews))

# Check whether review_id + order_id is unique
print(
    "Duplicate review_id + order_id combinations:",
    reviews_clean.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

# Check how many different orders each review ID is associated with
print(
    "Review IDs linked to multiple orders:",
    (duplicate_reviews.groupby("review_id")["order_id"].nunique() > 1).sum()
)

# Display sample records
duplicate_reviews[["review_id", "order_id", "review_score"]].head(10)

Records with repeated review IDs: 1603
Duplicate review_id + order_id combinations: 0
Review IDs linked to multiple orders: 789


,review_id,order_id,review_score
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1


In [8]:
# Check repeated order IDs in Reviews
print(
    "Orders with multiple review records:",
    reviews_clean.duplicated(subset=["order_id"]).sum()
)

# Check the maximum number of review records per order
print(
    "Maximum reviews per order:",
    reviews_clean.groupby("order_id").size().max()
)

Orders with multiple review records: 551
Maximum reviews per order: 3


In [9]:
foreign_key_checks = {
    "Orders -> Customers": (
        orders_clean["customer_id"],
        customers_clean["customer_id"]
    ),
    "Order Items -> Orders": (
        order_items_clean["order_id"],
        orders_clean["order_id"]
    ),
    "Order Items -> Products": (
        order_items_clean["product_id"],
        products_clean["product_id"]
    ),
    "Order Items -> Sellers": (
        order_items_clean["seller_id"],
        sellers_clean["seller_id"]
    ),
    "Payments -> Orders": (
        payments_clean["order_id"],
        orders_clean["order_id"]
    ),
    "Reviews -> Orders": (
        reviews_clean["order_id"],
        orders_clean["order_id"]
    )
}

for relationship, (child, parent) in foreign_key_checks.items():
    unmatched = (~child.isin(parent)).sum()
    print(relationship, "- Unmatched records:", unmatched)

Orders -> Customers - Unmatched records: 0
Order Items -> Orders - Unmatched records: 0
Order Items -> Products - Unmatched records: 0
Order Items -> Sellers - Unmatched records: 0
Payments -> Orders - Unmatched records: 0
Reviews -> Orders - Unmatched records: 0


In [10]:
# Display cleaned dataset data types

for name, df in {
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Customers": customers_clean,
    "Products": products_clean,
    "Sellers": sellers_clean,
    "Payments": payments_clean,
    "Reviews": reviews_clean,
    "Geolocation": geolocation_clean,
    "Category Translation": category_translation_clean
}.items():

    print("\n", name)
    print(df.dtypes)


 Orders
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

 Order Items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

 Customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

 Products
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_we

In [11]:
print("Customers ZIP length:")
print(customers_clean["customer_zip_code_prefix"].astype(str).str.len().value_counts())

print("\nSellers ZIP length:")
print(sellers_clean["seller_zip_code_prefix"].astype(str).str.len().value_counts())

print("\nGeolocation ZIP length:")
print(geolocation_clean["geolocation_zip_code_prefix"].astype(str).str.len().value_counts())

Customers ZIP length:
customer_zip_code_prefix
5    75446
4    23995
Name: count, dtype: int64

Sellers ZIP length:
seller_zip_code_prefix
5    2068
4    1027
Name: count, dtype: int64

Geolocation ZIP length:
geolocation_zip_code_prefix
5    579500
4    158832
Name: count, dtype: int64


In [12]:
customers_clean = pd.read_csv(
    os.path.join(PROCESSED_PATH, "customers_clean.csv"),
    dtype={"customer_zip_code_prefix": "string"}
)

sellers_clean = pd.read_csv(
    os.path.join(PROCESSED_PATH, "sellers_clean.csv"),
    dtype={"seller_zip_code_prefix": "string"}
)

geolocation_clean = pd.read_csv(
    os.path.join(PROCESSED_PATH, "geolocation_clean.csv"),
    dtype={"geolocation_zip_code_prefix": "string"}
)

In [13]:
print("Customers ZIP length:")
print(customers_clean["customer_zip_code_prefix"].str.len().value_counts())

print("\nSellers ZIP length:")
print(sellers_clean["seller_zip_code_prefix"].str.len().value_counts())

print("\nGeolocation ZIP length:")
print(geolocation_clean["geolocation_zip_code_prefix"].str.len().value_counts())

Customers ZIP length:
customer_zip_code_prefix
5    99441
Name: count, dtype: Int64

Sellers ZIP length:
seller_zip_code_prefix
5    3095
Name: count, dtype: Int64

Geolocation ZIP length:
geolocation_zip_code_prefix
5    738332
Name: count, dtype: Int64


In [14]:
# Order Items
print("Zero freight records:",
      (order_items_clean["freight_value"] == 0).sum())

# Products
print("Zero product weight records:",
      (products_clean["product_weight_g"] == 0).sum())

# Payments
print("Zero payment records:",
      (payments_clean["payment_value"] == 0).sum())

print("Zero installment records:",
      (payments_clean["payment_installments"] == 0).sum())

# Orders
invalid_delivery = orders_clean[
    pd.to_datetime(orders_clean["order_delivered_customer_date"]) <
    pd.to_datetime(orders_clean["order_delivered_carrier_date"])
]

print("Inconsistent delivery timestamp records:",
      len(invalid_delivery))

Zero freight records: 383
Zero product weight records: 4
Zero payment records: 9
Zero installment records: 2
Inconsistent delivery timestamp records: 23


In [15]:
# Approximate Brazil boundary screening
suspicious_geo = geolocation_clean[
    (geolocation_clean["geolocation_lat"] < -35) |
    (geolocation_clean["geolocation_lat"] > 6) |
    (geolocation_clean["geolocation_lng"] < -75) |
    (geolocation_clean["geolocation_lng"] > -30)
]

print("Suspicious geographical records:", len(suspicious_geo))

# ZIP prefixes associated with multiple states
multiple_states = (
    geolocation_clean.groupby("geolocation_zip_code_prefix")
    ["geolocation_state"].nunique()
)

print(
    "ZIP prefixes associated with multiple states:",
    (multiple_states > 1).sum()
)

Suspicious geographical records: 25
ZIP prefixes associated with multiple states: 8
